# 06 — Holon Subtypes and Classification

The CGA ontology declares six functional holon subtypes and two
lifecycle subtypes. Each carries specific semantics about what
belongs in its four layers and how it participates in the holarchy.

This notebook exercises three subtypes with dedicated SHACL shapes:

- `cga:AgentHolon` — an active computational agent whose interior
  is state/memory, boundary is output constraints, and context is
  execution history
- `cga:AggregateHolon` — a materialized view with no original data;
  interiors populated entirely by portal traversals
- `cga:AlignmentHolon` — vocabulary mappings between ontologies
  (covered in depth in notebook 04)

The SHACL shapes enforce per-subtype structural expectations at
Warning or Info severity. They catch misclassified holons early
without blocking valid experimentation.

In [ ]:
from holonic import HolonicDataset
import pyshacl

ds = HolonicDataset(load_ontology=True)

def validate(ds, label=""):
    registry = ds.backend.get_graph(ds.registry_iri)
    shapes = ds.backend.get_graph("urn:holonic:ontology:cga-shapes")
    conforms, _, report = pyshacl.validate(registry, shacl_graph=shapes)
    if label:
        print(f"=== {label} ===")
    print(f"Conforms: {conforms}")
    if not conforms:
        for line in report.split("\n"):
            if "Message" in line or "Severity" in line or "Focus" in line:
                print(f"  {line.strip()}")
    print()
    return conforms

## AgentHolon — active computational agent

An agent holon should have all four layers populated:
- Interior: agent state, memory, configuration
- Boundary: output constraints (what the agent is allowed to produce)
- Context: execution history (prov:Activity records)
- Projection: capability interface (what the agent offers to others)

The `AgentHolonShape` nudges (Info severity) when any layer is missing.

In [ ]:
# A well-formed agent holon with all four layers
ds.add_holon("urn:holon:classifier-agent", "Track Classifier v2",
             holon_type="cga:AgentHolon")

ds.add_interior("urn:holon:classifier-agent", '''
    @prefix cga:    <urn:holonic:ontology:> .
    @prefix schema: <https://schema.org/> .

    <urn:agent:classifier> a cga:AgentHolon ;
        schema:name "Track Classifier v2" ;
        schema:version "2.1.0" .

    <urn:config:threshold> a schema:PropertyValue ;
        schema:name "confidence_threshold" ;
        schema:value "0.85" .
''')

ds.add_boundary("urn:holon:classifier-agent", '''
    @prefix sh:     <http://www.w3.org/ns/shacl#> .
    @prefix schema: <https://schema.org/> .
    @prefix xsd:    <http://www.w3.org/2001/XMLSchema#> .

    <urn:shapes:ClassificationOutput> a sh:NodeShape ;
        sh:targetClass schema:CategoryCode ;
        sh:property [ sh:path schema:name ;  sh:minCount 1 ; sh:datatype xsd:string ] ;
        sh:property [ sh:path schema:value ; sh:minCount 1 ; sh:datatype xsd:decimal ] .
''')

ds.add_context("urn:holon:classifier-agent", '''
    @prefix prov: <http://www.w3.org/ns/prov#> .
    @prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

    <urn:run:2026-04-22> a prov:Activity ;
        prov:startedAtTime "2026-04-22T10:00:00Z"^^xsd:dateTime ;
        prov:endedAtTime   "2026-04-22T10:05:00Z"^^xsd:dateTime ;
        prov:wasAssociatedWith <urn:agent:classifier> .
''')

validate(ds, "Well-formed AgentHolon")

In [ ]:
# An agent holon missing its boundary and context layers
ds.add_holon("urn:holon:bare-agent", "Bare Agent",
             holon_type="cga:AgentHolon")
ds.add_interior("urn:holon:bare-agent", '''
    <urn:state:x> a <urn:AgentState> .
''')

validate(ds, "AgentHolon missing boundary + context")

## AggregateHolon — materialized view

An aggregate holon contains no original data. Its interiors are
populated entirely by portal traversals. The `AggregateHolonShape`
warns when an aggregate has interior data but no provenance
activity linking that data to a traversal.

This catches holons that were misclassified as aggregates when
they actually contain original data — or aggregates whose
provenance was lost.

In [ ]:
# An aggregate holon that was populated by a traversal (correct)
ds.add_holon("urn:holon:ops-rollup", "Ops Rollup",
             holon_type="cga:AggregateHolon")

interior_iri = "urn:holon:ops-rollup/interior/rollup"
ds.add_interior("urn:holon:ops-rollup", '''
    @prefix schema: <https://schema.org/> .
    <urn:rollup:1> a schema:Event ;
        schema:name "Aggregated event" .
''', graph_iri=interior_iri)

# Add provenance linking the interior to a traversal activity
ds.add_context("urn:holon:ops-rollup", f'''
    @prefix prov: <http://www.w3.org/ns/prov#> .
    @prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

    <urn:traversal:001> a prov:Activity ;
        prov:generated <{interior_iri}> ;
        prov:startedAtTime "2026-04-22T12:00:00Z"^^xsd:dateTime .
''')

validate(ds, "AggregateHolon with traversal provenance (correct)")

In [ ]:
# An aggregate holon with data but NO traversal provenance (suspect)
ds.add_holon("urn:holon:bad-aggregate", "Misclassified Aggregate",
             holon_type="cga:AggregateHolon")
ds.add_interior("urn:holon:bad-aggregate", '''
    <urn:raw:data> a <urn:OriginalRecord> ;
        <urn:field> "This was manually inserted, not traversed" .
''')
# No context graph with prov:Activity linking to this interior

validate(ds, "AggregateHolon with data but no provenance (should warn)")

## ClassificationLevel enum

Since 0.4.3, `cga:dataClassification` is an ObjectProperty whose
range is `cga:ClassificationLevel`. The ontology ships five standard
individuals: `Public`, `CUI`, `PII`, `Secret`, `TopSecret`.

This replaces the pre-0.4.3 free-text string approach, which let
typos and nonsense values through silently.

In [ ]:
# Holon with a valid classification
ds.add_holon("urn:holon:classified", "Classified Data",
             holon_type="cga:DataHolon")
ds.add_interior("urn:holon:classified", '''
    @prefix cga: <urn:holonic:ontology:> .
    <urn:holon:classified> cga:dataClassification cga:Secret ;
        cga:dataSteward <urn:person:j-chen> .
''')

validate(ds, "DataHolon with valid ClassificationLevel")

## Where to go next

- `05_governed_boundaries` — governance vocabulary in action:
  alignment holons, stewardship, classification boundary enforcement
- `docs/source/ontology.md` — full class and property reference
  including the portal subtype semantics table
- `docs/SPEC.md` OQ10 — upper-ontology alignment strategy
  (BFO/CCO, gist)